# Training Series: Introduction to Discontinuous Galerkin Methods for Flow Problems <br> - Hands-On Worksheets

In [1]:
//#r "../BoSSSpad/BoSSSpad.dll"
#r "C:\Program Files (x86)\FDY\BoSSS\bin\Release\net6.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Platform.LinAlg;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

## Hands-On 1: DG Basics - Projection and Approximation Properties 

### Problem statement

First, we define two functions: $g_1$ is continuous, $g_2$ has a discontinuity at $x = \pi$ in the first derivative

>$$ g_1(x) := \sin(x), $$
>$$g_2(x) := \vert \sin(x) \vert ,$$

The function argument is a vector $x \in R^n$, consisting only of one entry 
since we are working in a one dimensional space. 

**BoSSS** however supports 1D, 2D and 3D, so the spatial coordinate 
is a general vector.


### Plotting the functions

First, we plot the functions that are defined above over the interval $(0 ,2 \pi)$ with 
1000 sampling points using a Gnuplot-object.

In [2]:
// continuous, smooth  
Func<double[],double> g1 = (X => Math.Sin(X[0]));          
// continuous, non-smooth      
// == TODO == define g2  
Func<double[],double> g2 = (X => Math.Abs(Math.Sin(X[0])));          

We define equidistant sampling points...

In [3]:
double[] x = GenericBlas.Linspace(0, 2.0*Math.PI, 1000);

...and compute the function values. In the loop, we have to convert the scalar **x[i]** into an array
with one element, since $g_1$ has to be feed with arrays.

In [4]:
double[] g1_values = new double[x.Length];
for(int i = 0; i < x.Length; i++) {
    g1_values[i] = g1(new[] { x[i]} );
}

In [5]:
/// Instead of loops, we can also use Linq-functions:
double[] g2_values = x.Select(x => g2(new []{ x })).ToArray();

For now, we are using the simple plotting interface, which supports
Matlab-Style format specifiers and color names. 

In [6]:
Plot(X1:x, Y1:g1_values, Name1:"function g1", Format1:"--red",
     X2:x, Y2:g2_values, Name2:"function g2", Format2:"-.blue")

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 -1 
 
 
 
 
 -0.8 
 
 
 
 
 -0.6 
 
 
 
 
 -0.4 
 
 
 
 
 -0.2 
 
 
 
 
 0 
 
 
 
 
 0.2 
 
 
 
 
 0.4 
 
 
 
 
 0.6 
 
 
 
 
 0.8 
 
 
 
 
 1 
 
 
 
 
 0 
 
 
 
 
 1 
 
 
 
 
 2 
 
 
 
 
 3 
 
 
 
 
 4 
 
 
 
 
 5 
 
 
 
 
 6 
 
 
 
 
 7 
 
 
 
 
 
 
 
 
 function g1 
 
 
 function g1 
 
 
 
	<path stroke='rgb(255, 0, 0)' stroke-dasharray='2.5,4.0' d='M716.2,34.7 L758.4,34.7 M53.9,290.3 L54.5,288.6 L55.2,286.9 L55.8,285.2 L56.5,283.5 L57.1,281.7
 L57.8,280.0 L58.4,278.3 L59.1,276.6 L59.7,274.9 L60.4,273.2 L61.0,271.4 L61.7,269.7 L62.3,268.0
 L63.0,266.3 L63.6,264.6 L64.3,262.9 L64.9,261.1 L65.6,259.4 L66.2,257.7 L66.9,256.0 L67.5,254.3
 L68.2,252.6 L68.8,250.9 L69.4,249.2 L70.1,247.5 L70.7,245.8 L71.4,244.1 L72.0,242.4 L72.7,240.7
 L73.3,239.0 L74.0,237.3 L74.6,235.6 L75.3,234.0 L75.9,232.3 L76.6,230.6 L77.2,228.9 L77.9,227.2
 L78.5,225.6 L79.2,223.9 L79.8,222.2 L80.5,220.6 L81.1,218.9 L81.8,217.2 L82.4,215.6 L83.1,213.9
 L83.7,212.3 L84.4,210.6 L85.0,209.0 L85.6,207.3 L86.3,205.7 L86.9,204.1 L87.6,202.4 L88.2,200.8
 L88.9,199.2 L89.5,197.6 L90.2,195.9 L90.8,194.3 L91.5,192.7 L92.1,191.1 L92.8,189.5 L93.4,187.9
 L94.1,186.3 L94.7,184.7 L95.4,183.1 L96.0,181.6 L96.7,180.0 L97.3,178.4 L98.0,176.8 L98.6,175.3
 L99.3,173.7 L99.9,172.2 L100.5,170.6 L101.2,169.1 L101.8,167.5 L102.5,166.0 L103.1,164.5 L103.8,162.9
 L104.4,161.4 L105.1,159.9 L105.7,158.4 L106.4,156.9 L107.0,155.4 L107.7,153.9 L108.3,152.4 L109.0,150.9
 L109.6,149.4 L110.3,148.0 L110.9,146.5 L111.6,145.0 L112.2,143.6 L112.9,142.1 L113.5,140.7 L114.2,139.3
 L114.8,137.8 L115.5,136.4 L116.1,135.0 L116.7,133.6 L117.4,132.2 L118.0,130.8 L118.7,129.4 L119.3,128.0
 L120.0,126.6 L120.6,125.2 L121.3,123.8 L121.9,122.5 L122.6,121.1 L123.2,119.8 L123.9,118.4 L124.5,117.1
 L125.2,115.8 L125.8,114.5 L126.5,113.1 L127.1,111.8 L127.8,110.5 L128.4,109.2 L129.1,107.9 L129.7,106.7
 L130.4,105.4 L131.0,104.1 L131.6,102.9 L132.3,101.6 L132.9,100.4 L133.6,99.1 L134.2,97.9 L134.9,96.7
 L135.5,95.5 L136.2,94.3 L136.8,93.1 L137.5,91.9 L138.1,90.7 L138.8,89.5 L139.4,88.4 L140.1,87.2
 L140.7,86.1 L141.4,84.9 L142.0,83.8 L142.7,82.7 L143.3,81.6 L144.0,80.4 L144.6,79.3 L145.3,78.3
 L145.9,77.2 L146.6,76.1 L147.2,75.0 L147.8,74.0 L148.5,72.9 L149.1,71.9 L149.8,70.8 L150.4,69.8
 L151.1,68.8 L151.7,67.8 L152.4,66.8 L153.0,65.8 L153.7,64.8 L154.3,63.9 L155.0,62.9 L155.6,62.0
 L156.3,61.0 L156.9,60.1 L157.6,59.2 L158.2,58.2 L158.9,57.3 L159.5,56.4 L160.2,55.5 L160.8,54.7
 L161.5,53.8 L162.1,52.9 L162.7,52.1 L163.4,51.2 L164.0,50.4 L164.7,49.6 L165.3,48.8 L166.0,48.0
 L166.6,47.2 L167.3,46.4 L167.9,45.6 L168.6,44.8 L169.2,44.1 L169.9,43.3 L170.5,42.6 L171.2,41.9
 L171.8,41.2 L172.5,40.5 L173.1,39.8 L173.8,39.1 L174.4,38.4 L175.1,37.7 L175.7,37.1 L176.4,36.4
 L177.0,35.8 L177.6,35.2 L178.3,34.6 L178.9,33.9 L179.6,33.4 L180.2,32.8 L180.9,32.2 L181.5,31.6
 L182.2,31.1 L182.8,30.5 L183.5,30.0 L184.1,29.5 L184.8,28.9 L185.4,28.4 L186.1,28.0 L186.7,27.5
 L187.4,27.0 L188.0,26.5 L188.7,26.1 L189.3,25.6 L190.0,25.2 L190.6,24.8 L191.3,24.4 L191.9,24.0
 L192.6,23.6 L193.2,23.2 L193.8,22.8 L194.5,22.5 L195.1,22.1 L195.8,21.8 L196.4,21.5 L197.1,21.2
 L197.7,20.9 L198.4,20.6 L199.0,20.3 L199.7,20.0 L200.3,19.7 L201.0,19.5 L201.6,19.3 L202.3,19.0
 L202.9,18.8 L203.6,18.6 L204.2,18.4 L204.9,18.2 L205.5,18.0 L206.2,17.9 L206.8,17.7 L207.5,17.6
 L208.1,17.4 L208.7,17.3 L209.4,17.2 L210.0,17.1 L210.7,17.0 L211.3,16.9 L212.0,16.9 L212.6,16.8
 L213.3,16.8 L213.9,16.7 L214.6,16.7 L215.2,16.7 L215.9,16.7 L216.5,16.7 L217.2,16.7 L217.8,16.8
 L218.5,16.8 L219.1,16.8 L219.8,16.9 L220.4,17.0 L221.1,17.1 L221.7,17.2 L222.4,17.3 L223.0,17.4
 L223.7,17.5 L224.3,17.6 L224.9,17.8 L225.6,18.0 L226.2,18.1 L226.9,18.3 L227.5,18.5 L228.2,18.7
 L228.8,18.9 L229.5,19

### Projection onto the DG space

Next, we create a grid which has a cell boundary exactly at the position of
the discontinuity of $g_2$.

In [7]:
var Nodes1 = new double[] {0, 2, Math.PI, 4.5, 2*Math.PI };         
var Grid1 = Grid1D.LineGrid(Nodes1);

We can get the total number of cells by using the following command:

In [8]:
Grid1.NumberOfCells

4

The recently created grid-object is not directly usable because it contains only the nodes of the grid. 

We have to create a **GridData**-object which provides all necessary transformation metrics, etc. .


In [9]:
var gdata1 = new GridData(Grid1);

At this point, we are able to create the so-called ***DG fields*** to approximate ***$g_1$***
on ***grid1***. 

Therefore, we project ***$g_1$*** onto ***grid1*** using polynomial orders
of $p=2$ and $p=8$.

In [46]:
var g1_grid1_p2 = new SinglePhaseField(new Basis(gdata1, 2), "g1 with p2 at Grid 1");      
g1_grid1_p2.ProjectField(g1);

In [11]:
// == TODO == project g1 on grid1 with p=8
var g1_grid1_p8 = new SinglePhaseField(new Basis(gdata1, 8), "g1 with p8 at Grid 1");         
g1_grid1_p8.ProjectField(g1);

Now, let us plot the projected solution for $p=2$. 

By using the upsampling parameter, we can determine 
the amount of sampling points per cell.


In [12]:
var upsampling = 20;

In [47]:
var gp1 = new Gnuplot();

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


In [48]:
gp1.PlotField(g1_grid1_p2,          
    new PlotFormat(lineColor: (LineColors)(1)),
    upsampling);

In [49]:
gp1.PlotNow() // shows the plot

Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 -1.2 
 
 
 
 
 -1 
 
 
 
 
 -0.8 
 
 
 
 
 -0.6 
 
 
 
 
 -0.4 
 
 
 
 
 -0.2 
 
 
 
 
 0 
 
 
 
 
 0.2 
 
 
 
 
 0.4 
 
 
 
 
 0.6 
 
 
 
 
 0.8 
 
 
 
 
 1 
 
 
 
 
 0 
 
 
 
 
 1 
 
 
 
 
 2 
 
 
 
 
 3 
 
 
 
 
 4 
 
 
 
 
 5 
 
 
 
 
 6 
 
 
 
 
 7 
 
 
 
 
 
 
 
 
 g1 with p2 at Grid 1 
 
 
 g1 with p2 at Grid 1 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M716.2,34.7 L758.4,34.7 M53.9,275.7 L64.7,243.5 L75.6,213.4 L86.4,185.5 L97.3,159.8 L108.1,136.2
 L119.0,114.8 L129.8,95.6 L140.6,78.5 L151.5,63.5 L162.3,50.7 L173.2,40.1 L184.0,31.6 L194.9,25.3
 L205.7,21.2 L216.6,19.2 L227.4,19.4 L238.2,21.7 L249.1,26.2 L259.9,32.8 L259.9,36.9 L266.1,44.8
 L272.3,53.1 L278.5,62.0 L284.7,71.3 L290.9,81.1 L297.1,91.4 L303.3,102.2 L309.4,113.4 L315.6,125.1
 L321.8,137.3 L328.0,149.9 L334.2,163.0 L340.4,176.6 L346.6,190.7 L352.8,205.2 L359.0,220.2 L365.1,235.7
 L371.3,251.7 L377.5,268.1 L377.5,261.3 L384.9,281.4 L392.3,300.8 L399.6,319.4 L407.0,337.3 L414.4,354.4
 L421.7,370.7 L429.1,386.2 L436.4,400.9 L443.8,414.9 L451.2,428.1 L458.5,440.6 L465.9,452.2 L473.3,463.1
 L480.6,473.3 L488.0,482.6 L495.4,491.2 L502.7,499.0 L510.1,506.0 L517.5,512.3 L517.5,514.5 L527.1,515.5
 L536.8,514.8 L546.5,512.6 L556.1,508.7 L565.8,503.2 L575.5,496.1 L585.1,487.4 L594.8,477.1 L604.5,465.2
 L614.1,451.6 L623.8,436.5 L633.5,419.7 L643.1,401.4 L652.8,381.4 L662.5,359.8 L672.2,336.6 L681.8,311.8
 L691.5,285.4 L701.2,257.4 '/>

### Computing the $L^2$-error

Next, we learn how to compute the $L^2$-error for both approximations of $g_1$ with different polynomial degrees:

In [16]:
g1_grid1_p2.L2Error(g1)

0.004232254221060506

In [17]:
// == TODO == L2 error for p=8
g1_grid1_p8.L2Error(g1)

4.039445228471717E-10

### Plotting the point-wise error

Now, we plot the point-wise error for the approximation of $g_1$ 
on **grid1** with a polynomial degree of 8.

In [18]:
int K = 20; // number of points per cell == TODO == try higher numbers        
var gp2 = new Gnuplot();         
gp2.PlotLogError(g1_grid1_p8, g1, "g1 with p8 at Grid 1", K,          
    new PlotFormat(lineColor: (LineColors)(1)));         
gp2.PlotNow()

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 1x10 -11 
 
 
 
 
 1x10 -10 
 
 
 
 
 1x10 -9 
 
 
 
 
 1x10 -8 
 
 
 
 
 1x10 -7 
 
 
 
 
 0 
 
 
 
 
 1 
 
 
 
 
 2 
 
 
 
 
 3 
 
 
 
 
 4 
 
 
 
 
 5 
 
 
 
 
 6 
 
 
 
 
 7 
 
 
 
 
 
 
 
 
 g1 with p8 at Grid 1 
 
 
 g1 with p8 at Grid 1 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M716.2,34.7 L758.4,34.7 M78.8,123.7 L89.3,187.8 L99.7,209.9 L110.2,213.8 L120.7,247.1 L131.1,202.6
 L141.6,263.6 L152.1,233.5 L162.6,207.7 L173.0,260.8 L183.5,244.2 L194.0,208.6 L204.4,249.0 L214.9,249.0
 L225.4,206.9 L235.8,270.5 L246.3,214.6 L256.8,223.3 L267.2,194.1 L277.7,133.5 L277.7,402.0 L283.7,464.2
 L289.7,490.3 L295.6,487.0 L301.6,532.9 L307.6,477.9 L313.6,526.9 L319.5,515.2 L325.5,481.0 L331.5,522.7
 L337.5,526.6 L343.4,480.8 L349.4,511.6 L355.4,530.4 L361.4,476.9 L367.4,527.4 L373.3,486.8 L379.3,487.2
 L385.3,462.7 L391.3,399.7 L391.3,311.0 L398.4,374.2 L405.5,398.3 L412.6,398.6 L419.7,438.1 L426.8,388.5
 L433.9,443.0 L441.0,422.6 L448.1,392.5 L455.3,439.4 L462.4,433.6 L469.5,392.8 L476.6,428.0 L483.7,437.9
 L490.8,390.0 L497.9,446.2 L505.0,398.9 L512.1,403.0 L519.2,376.4 L526.4,314.5 L526.4,184.0 L535.7,245.3
 L545.0,273.2 L554.4,266.6 L563.7,318.5 L573.0,258.4 L582.4,303.1 L591.7,298.6 L601.0,260.6 L610.4,298.5
 L619.7,310.2 L629.0,260.0 L638.4,287.7 L647.7,313.4 L657.0,255.4 L666.4,302.1 L675.7,266.1 L685.0,263.8
 L694.4,240.8 L703.7,177.1 '/>

### Decay behavior of the DG modes for smooth and non-smooth functions

We investigate the decay behavior of the DG modes for both smooth and non-smooth functions. 

For this purpose, we create a second grid which has the
discontinuity of $g_2$ within a cell and project $g_2$ onto this grid
like mentioned above.

In [19]:
// == TODO == define the following variables
double[] Nodes2 = new double[] {0, 2, 4.5, 2*Math.PI };         
Grid1D Grid2 = Grid1D.LineGrid(Nodes2);         
GridData gdata2 = new GridData(Grid2);         
SinglePhaseField g2_grid2_p8 = new SinglePhaseField(new Basis(gdata2, 8), "g2_p8 at Grid2");         
g2_grid2_p8.ProjectField(g2);

The cell coordinates can be extracted by using the **Coordinates** parameter.

In [20]:
double[] cell0 = g2_grid2_p8.Coordinates.GetRow(0);          
    // coord. in cell 0 (smooth)

In [21]:
double[] cell1 = g2_grid2_p8.Coordinates.GetRow(1);          
    // coord. in cell 1 (with kink)

In [22]:
double[] cell2 = g2_grid2_p8.Coordinates.GetRow(2);          
    // coord. in cell 2 (smooth)

Only the absolute value shall be plotted. We use a for-loop to replace the data in 
**cell0**, **cell1** and **cell2** by their absolute values.


In [23]:
for(int i = 0; i < cell0.Length; i++) {         
    cell0[i] = Math.Abs(cell0[i]);         
    cell1[i] = Math.Abs(cell1[i]);         
    cell2[i] = Math.Abs(cell2[i]);         
 }

In [24]:
Plot(X1:null, Y1:cell1, Name1:"disc. cell", Format1:"*-magenta",
     X2:null, Y2:cell0, Name2:"cell0",      Format2:"o-blue",
     X3:null, Y3:cell2, Name3:"cell2",      Format3:"o-red")

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 0.2 
 
 
 
 
 0.4 
 
 
 
 
 0.6 
 
 
 
 
 0.8 
 
 
 
 
 1 
 
 
 
 
 1.2 
 
 
 
 
 0 
 
 
 
 
 1 
 
 
 
 
 2 
 
 
 
 
 3 
 
 
 
 
 4 
 
 
 
 
 5 
 
 
 
 
 6 
 
 
 
 
 7 
 
 
 
 
 8 
 
 
 
 
 
 
 
 
 disc. cell 
 
 
 disc. cell 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 cell0 
 
 
 cell0 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 cell2 
 
 
 cell2

#### Note
Using a shortcut for the for-loop above, the absolute values in **cell0**
can also be stored using the following command:

In [25]:
double[] cell0 = g2_grid2_p8.Coordinates.GetRow(0)         
    .Select(d => Math.Abs(d)).ToArray();

Now, we would like to plot the logarithm (use **Math.Log10(...)**) of the absolute
values of the DG coordinates.

In [26]:
Plot(X1:null, Y1:cell1, Name1:"disc. cell", Format1:"*-magenta",
     X2:null, Y2:cell0, Name2:"cell0",      Format2:"o-blue",
     X3:null, Y3:cell2, Name3:"cell2",      Format3:"o-red",
     logY:true)

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.00000001 
 
 
 
 
 0.00000010 
 
 
 
 
 0.00000100 
 
 
 
 
 0.00001000 
 
 
 
 
 0.00010000 
 
 
 
 
 0.00100000 
 
 
 
 
 0.01000000 
 
 
 
 
 0.10000000 
 
 
 
 
 1.00000000 
 
 
 
 
 10.00000000 
 
 
 
 
 0 
 
 
 
 
 1 
 
 
 
 
 2 
 
 
 
 
 3 
 
 
 
 
 4 
 
 
 
 
 5 
 
 
 
 
 6 
 
 
 
 
 7 
 
 
 
 
 8 
 
 
 
 
 
 
 
 
 disc. cell 
 
 
 disc. cell 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 cell0 
 
 
 cell0 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 cell2 
 
 
 cell2

### Convergence study

In this section, we learn how to perform a convergence study for $g_2$
for two different sequences of grid resolutions and different polynomial
orders. 

Therefore, we define two different sequences of grid resolutions:

In [27]:
int[][] ResSeq = new int[2][];

Grid resolutions so that the kink in **g2** is located at a cell boundary:

In [28]:
ResSeq[0] = new int[] { 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048 };

Grid resolutions so that the kink in **g2** is located within a cell:

In [29]:
// == TODO == for ResSeq[1] set a int-array with following entries: 3, 7, 15, 31, 63, 127, 255, 511, 1023, 2047
ResSeq[1] = new int[] { 3, 7, 15, 31, 63, 127, 255, 511, 1023, 2047 };

We save our errors into a multidimensional array by looping over 

- the resolution sequence
- the polynomial order
- the resolution

In [30]:
var Errors = MultidimensionalArray.Create(2, 5, ResSeq[0].Length);

In [31]:
for(int i = 0; i < 2; i++) { // loop over the resolution sequence         
    for(int p = 0; p <= 4; p++) { // loop over polynomial orders         
        for(int k = 0; k < ResSeq[i].Length; k++) { // loop over different resolutions        
 
            Console.Write("polynomial order {1}"+         
            ",\tResolution {0}... ", ResSeq[i][k], p);         
 
            var grid  = Grid1D.LineGrid(GenericBlas.Linspace(0, 2.0*Math.PI, ResSeq[i][k] + 1));           
            // number of nodes == number of cells + 1         
 
            var gData = new GridData(grid);         
 
            var g2_h  = new SinglePhaseField(new Basis(gData, p));         
 
            // == TODO == What's missing? 
            g2_h.ProjectField(g2);         
 
            Errors[i,p,k] = g2_h.L2Error(g2);         
 
            Console.WriteLine("\tdone: L2 error is {0:0.###e-00}.", Errors[i,p,k]);         
        }         
    }         
 }

polynomial order 0,	Resolution 4... 	done: L2 error is 1.791e-01.
polynomial order 0,	Resolution 8... 	done: L2 error is 4.536e-02.
polynomial order 0,	Resolution 16... 	done: L2 error is 1.138e-02.
polynomial order 0,	Resolution 32... 	done: L2 error is 2.846e-03.
polynomial order 0,	Resolution 64... 	done: L2 error is 7.118e-04.
polynomial order 0,	Resolution 128... 	done: L2 error is 1.779e-04.
polynomial order 0,	Resolution 256... 	done: L2 error is 4.449e-05.
polynomial order 0,	Resolution 512... 	done: L2 error is 1.112e-05.
polynomial order 0,	Resolution 1024... 	done: L2 error is 2.781e-06.
polynomial order 0,	Resolution 2048... 	done: L2 error is 6.951e-07.
polynomial order 1,	Resolution 4... 	done: L2 error is 2.155e-02.
polynomial order 1,	Resolution 8... 	done: L2 error is 2.739e-03.
polynomial order 1,	Resolution 16... 	done: L2 error is 3.438e-04.
polynomial order 1,	Resolution 32... 	done: L2 error is 4.302e-05.
polynomial order 1,	Resolution 64... 	done: L2 error is 5.3

In [32]:
using NUnit.Framework;

In [33]:
/// NUnit test (few random tests)
Assert.LessOrEqual(Errors[1,4,9],8E-06);       
Assert.LessOrEqual(Errors[1,3,9],7.5E-06);       
Assert.LessOrEqual(Errors[1,2,9],2E-05);       
Assert.LessOrEqual(Errors[1,1,9],2E-05);       
Assert.LessOrEqual(Errors[0,3,9],1E-12);       
Assert.LessOrEqual(Errors[0,3,9],1E-12);       
Assert.LessOrEqual(Errors[1,4,0],0.25);       
Assert.LessOrEqual(Errors[0,0,0],0.2);       
Assert.LessOrEqual(Errors[0,3,0],1E-03);

We plot the error for the grids which have the kink at the cell boundary,
there we reach spectral convergence:

In [34]:
var hValues = ResSeq[0].Select(J => Math.PI*2.0/J);
Plot(X1:hValues, Y1:Errors.ExtractSubArrayShallow(0,0,-1).To1DArray(),
     Name1:"grid1,p0", Format1:"o-red",
     X2:hValues, Y2:Errors.ExtractSubArrayShallow(0,1,-1).To1DArray(),
     Name2:"grid1,p1", Format2:"o-blue",
     X3:hValues, Y3:Errors.ExtractSubArrayShallow(0,2,-1).To1DArray(),
     Name3:"grid1,p2", Format3:"o-green",
     X4:hValues, Y4:Errors.ExtractSubArrayShallow(0,3,-1).To1DArray(),
     Name4:"grid1,p3", Format4:"o-magenta",
     X5:hValues, Y5:Errors.ExtractSubArrayShallow(0,4,-1).To1DArray(),
     Name5:"grid1,p4", Format5:"o-orange",
     logX:true, logY:true)

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 1x10 -16 
 
 
 
 
 1x10 -14 
 
 
 
 
 1x10 -12 
 
 
 
 
 1x10 -10 
 
 
 
 
 1x10 -8 
 
 
 
 
 1x10 -6 
 
 
 
 
 0.0001 
 
 
 
 
 0.01 
 
 
 
 
 1 
 
 
 
 
 0.001 
 
 
 
 
 0.01 
 
 
 
 
 0.1 
 
 
 
 
 1 
 
 
 
 
 10 
 
 
 
 
 
 
 
 
 grid1,p0 
 
 
 grid1,p0 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 grid1,p1 
 
 
 grid1,p1 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 grid1,p2 
 
 
 grid1,p2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 grid1,p3 
 
 
 grid1,p3 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 grid1,p4 
 
 
 grid1,p4

Now we plot the error for the grids which have the kink within a cell;
due to the low regularity, the convergence of the DG method
degenerates:

In [35]:
var hValues = ResSeq[0].Select(J => Math.PI*2.0/J);
Plot(X1:hValues, Y1:Errors.ExtractSubArrayShallow(1,0,-1).To1DArray(),
     Name1:"grid1,p0", Format1:"o-red",
     X2:hValues, Y2:Errors.ExtractSubArrayShallow(1,1,-1).To1DArray(),
     Name2:"grid1,p1", Format2:"o-blue",
     X3:hValues, Y3:Errors.ExtractSubArrayShallow(1,2,-1).To1DArray(),
     Name3:"grid1,p2", Format3:"o-green",
     X4:hValues, Y4:Errors.ExtractSubArrayShallow(1,3,-1).To1DArray(),
     Name4:"grid1,p3", Format4:"o-magenta",
     X5:hValues, Y5:Errors.ExtractSubArrayShallow(1,4,-1).To1DArray(),
     Name5:"grid1,p4", Format5:"o-orange",
     logX:true, logY:true)

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.0000010 
 
 
 
 
 0.0000100 
 
 
 
 
 0.0001000 
 
 
 
 
 0.0010000 
 
 
 
 
 0.0100000 
 
 
 
 
 0.1000000 
 
 
 
 
 1.0000000 
 
 
 
 
 0.001 
 
 
 
 
 0.01 
 
 
 
 
 0.1 
 
 
 
 
 1 
 
 
 
 
 10 
 
 
 
 
 
 
 
 
 grid1,p0 
 
 
 grid1,p0 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 grid1,p1 
 
 
 grid1,p1 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 grid1,p2 
 
 
 grid1,p2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 grid1,p3 
 
 
 grid1,p3 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 grid1,p4 
 
 
 grid1,p4